# Admin setup — run me once

This notebook is for **you** (the project lead), not your colleague. Run it to:

1. Create the `disney_lessons` dataset in your BigQuery project
2. Load the practice tables (currently `princesses` — more added as new lessons are released)
3. Grant your colleague read-only access to the dataset

You can run this notebook **before** you have a specific colleague in mind. Just leave `COLLEAGUE_EMAIL` blank in the configuration cell — the dataset and tables will still be set up, and the IAM steps will skip cleanly. When you're ready to onboard someone, come back, set the email, and rerun the relevant cells.

⏱️ ~2 minutes.

## Step 1 — Configuration

Edit the cell below to set:
- `PROJECT_ID` — your BigQuery project ID (already filled in for you).
- `COLLEAGUE_EMAIL` — your colleague's Google email. **Leave empty (`""`) if you're testing solo first.** You can fill this in later and rerun the notebook.

Then run the cell. The first time, a popup will ask you to sign in to Google.

In [ ]:
from google.colab import auth
auth.authenticate_user()

# === CONFIGURE ME ===
PROJECT_ID = "sql-with-disney"   # ← your BigQuery project
COLLEAGUE_EMAIL = ""           # ← leave empty for solo setup; set this when ready to onboard your colleague
# ====================

if PROJECT_ID in ("your-project-id-here", ""):
    raise ValueError("Set PROJECT_ID to your BigQuery Project ID.")

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
DATASET_ID = f"{PROJECT_ID}.disney_lessons"

print(f"✓ Connected to project: {PROJECT_ID}")
if COLLEAGUE_EMAIL:
    print(f"✓ Will grant access to: {COLLEAGUE_EMAIL}")
else:
    print("ℹ No COLLEAGUE_EMAIL set — IAM grant steps (4 and 5) will skip.")
    print("  Set COLLEAGUE_EMAIL when you're ready to onboard, then rerun those cells.")

## Step 2 — Create the dataset

Creates `disney_lessons` if it doesn't already exist. Re-running is safe (no-op).

In [ ]:
client.create_dataset(DATASET_ID, exists_ok=True)
print(f"✓ Dataset ready: {DATASET_ID}")

## Step 3 — Load the princesses table

Loads 16 rows of Disney princess data. Re-running overwrites the table (`WRITE_TRUNCATE`), so it's always idempotent.

In [ ]:
princesses = pd.DataFrame([
    {"name": "Snow White", "movie": "Snow White and the Seven Dwarfs", "year": 1937, "home": "Forest cottage"},
    {"name": "Cinderella", "movie": "Cinderella",                       "year": 1950, "home": "Kingdom of Charming"},
    {"name": "Aurora",     "movie": "Sleeping Beauty",                  "year": 1959, "home": "Forest cottage"},
    {"name": "Ariel",      "movie": "The Little Mermaid",               "year": 1989, "home": "Atlantica"},
    {"name": "Belle",      "movie": "Beauty and the Beast",             "year": 1991, "home": "Villeneuve"},
    {"name": "Jasmine",    "movie": "Aladdin",                          "year": 1992, "home": "Agrabah"},
    {"name": "Pocahontas", "movie": "Pocahontas",                       "year": 1995, "home": "Powhatan village"},
    {"name": "Mulan",      "movie": "Mulan",                            "year": 1998, "home": "China"},
    {"name": "Tiana",      "movie": "The Princess and the Frog",        "year": 2009, "home": "New Orleans"},
    {"name": "Rapunzel",   "movie": "Tangled",                          "year": 2010, "home": "Corona"},
    {"name": "Merida",     "movie": "Brave",                            "year": 2012, "home": "DunBroch"},
    {"name": "Anna",       "movie": "Frozen",                           "year": 2013, "home": "Arendelle"},
    {"name": "Elsa",       "movie": "Frozen",                           "year": 2013, "home": "Arendelle"},
    {"name": "Moana",      "movie": "Moana",                            "year": 2016, "home": "Motunui"},
    {"name": "Raya",       "movie": "Raya and the Last Dragon",         "year": 2021, "home": "Heart"},
    {"name": "Asha",       "movie": "Wish",                             "year": 2023, "home": "Rosas"},
])

client.load_table_from_dataframe(
    princesses,
    f"{DATASET_ID}.princesses",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
).result()

print(f"✓ Loaded princesses table ({len(princesses)} rows)")

## Step 4 — Grant your colleague read access (skipped if no email set)

If `COLLEAGUE_EMAIL` is set, this adds her as a **BigQuery Data Viewer** on the `disney_lessons` dataset. She'll be able to read the practice tables and nothing else.

If `COLLEAGUE_EMAIL` is empty, this cell skips with a message. Come back later, set the email in Step 1, and rerun this cell.

In [ ]:
if not COLLEAGUE_EMAIL:
    print("ℹ Skipping — COLLEAGUE_EMAIL is empty.")
    print("  Set it in Step 1 when you're ready to onboard, then rerun this cell.")
else:
    dataset = client.get_dataset(DATASET_ID)
    entries = list(dataset.access_entries)

    already_granted = any(
        e.entity_id == COLLEAGUE_EMAIL and e.role == "READER"
        for e in entries
    )

    if already_granted:
        print(f"  {COLLEAGUE_EMAIL} already has Data Viewer access — nothing to do.")
    else:
        entries.append(bigquery.AccessEntry(
            role="READER",
            entity_type="userByEmail",
            entity_id=COLLEAGUE_EMAIL,
        ))
        dataset.access_entries = entries
        client.update_dataset(dataset, ["access_entries"])
        print(f"✓ Granted Data Viewer to {COLLEAGUE_EMAIL} on disney_lessons")

## Step 5 — Grant the BigQuery Job User role (manual, ~30 seconds)

Your colleague also needs permission to **run** queries (vs. just read data). This is a one-time, click-through step in the GCP Console.

If `COLLEAGUE_EMAIL` is set, the cell below prints a direct link with the steps. Otherwise it skips.

In [ ]:
if not COLLEAGUE_EMAIL:
    print("ℹ Skipping — COLLEAGUE_EMAIL is empty.")
    print("  When you're ready to onboard a colleague:")
    print("    1. Set COLLEAGUE_EMAIL in Step 1")
    print("    2. Rerun cells 9 and 11")
else:
    print(f"Open this URL to grant the BigQuery Job User role:\n")
    print(f"  https://console.cloud.google.com/iam-admin/iam?project={PROJECT_ID}\n")
    print(f"Then:")
    print(f"  1. Click + GRANT ACCESS")
    print(f"  2. New principals: {COLLEAGUE_EMAIL}")
    print(f"  3. Role:           BigQuery Job User")
    print(f"  4. Click SAVE")

## Verify (optional)

Confirms the dataset and tables exist and shows who has dataset-level access.

In [ ]:
print(f"Tables in disney_lessons:")
for t in client.list_tables(DATASET_ID):
    print(f"  - {t.table_id}")

dataset = client.get_dataset(DATASET_ID)
print(f"\nDataset-level access on disney_lessons:")
for e in dataset.access_entries:
    role = e.role or "(no role)"
    who = e.entity_id
    print(f"  - {who}: {role}")

## Done. What's next?

If you set `COLLEAGUE_EMAIL` and ran all the steps:
1. Drop the lesson notebooks (`00_concepts.ipynb`, `01_first_query.ipynb`, ...) into a shared Google Drive folder
2. Share with your colleague
3. She opens, signs in, runs cells, learns SQL

If you ran solo (no email):
- Test the lesson notebooks yourself first — your owner role lets you query freely
- Come back here when you're ready to onboard, fill in `COLLEAGUE_EMAIL`, rerun cells 9 and 11

When new lessons are released that need additional tables (`movies`, `characters`, `parks`, `attractions`), come back and add the load steps.